# Notebook resolvido - Semana 9: Regressão Logística

**Objetivo:** Implementar um classificador de spam e entender as métricas de avaliação.

**Instruções:**
1. Baixar o notebook exemplo disponível no repositório

2. Executar o código de regressão logística(dataset de e-mails)

3. **Testar mudanças:**
* Alterar a proporção de treino/teste
* Mudar o limite de decisão(se quiser testar)

4. **Responder no notebook:**
* Qual foi a acurácia, precisão e recall obtidos?
* O que cada métrica significa neste contexto(spam)?
* Para um filtro de spam, qual métrica você acha mais importante? Porquê?

5. Subir o notebook respondido na pasta da semana 10 do repositório

Aluna: Kailayni Rodrigues Janez

# 1. Importar bibliotecas

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# Para reprodutibilidade
np.random.seed(42)

# 2. Carregar o dataset

In [8]:
# Baixar do site original da UCI (dados de SMS Spam Collection)
!wget https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip -O sms_spam.zip
!unzip -o sms_spam.zip

# O arquivo extraído se chama SMSSpamCollection
df = pd.read_csv('SMSSpamCollection', sep='\t', header=None, names=['label', 'message'])

print("Dataset carregado com sucesso!")
print(f"Total de mensagens: {len(df)}")
print(df.head())

--2026-06-08 23:39:11--  https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘sms_spam.zip’

sms_spam.zip            [ <=>                ] 198.65K  1.05MB/s    in 0.2s    

2026-06-08 23:39:12 (1.05 MB/s) - ‘sms_spam.zip’ saved [203415]

Archive:  sms_spam.zip
  inflating: SMSSpamCollection       
  inflating: readme                  
Dataset carregado com sucesso!
Total de mensagens: 5572
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro

# 3. Explorar os dados

In [9]:
# Verificar distribuição das classes
print("Distribuição das classes:")
print(df['label'].value_counts())
print(f"\nProporção de spam: {df['label'].value_counts()['spam'] / len(df) * 100:.2f}%")

# Converter rótulos para números (spam = 1, ham = 0)
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

print("\nExemplo de mensagem ham (não spam):")
print(df[df['label'] == 'ham']['message'].iloc[0])
print("\nExemplo de mensagem spam:")
print(df[df['label'] == 'spam']['message'].iloc[0])

Distribuição das classes:
label
ham     4825
spam     747
Name: count, dtype: int64

Proporção de spam: 13.41%

Exemplo de mensagem ham (não spam):
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

Exemplo de mensagem spam:
Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's


# 4. Converter texto em números (Bag of Words)

In [10]:
# Criar representação numérica das mensagens
# Cada palavra vira uma coluna
vectorizer = CountVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(df['message']).toarray()
y = df['label_num'].values

print(f"Matriz de características: {X.shape}")
print(f"Total de palavras consideradas: {len(vectorizer.get_feature_names_out())}")

Matriz de características: (5572, 5000)
Total de palavras consideradas: 5000


# 5. Separar treino e teste

In [11]:
# 80% treino, 20% teste
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Treino: {len(X_treino)} mensagens")
print(f"Teste: {len(X_teste)} mensagens")
print(f"Proporção de spam no treino: {y_treino.mean()*100:.2f}%")
print(f"Proporção de spam no teste: {y_teste.mean()*100:.2f}%")

Treino: 4457 mensagens
Teste: 1115 mensagens
Proporção de spam no treino: 13.42%
Proporção de spam no teste: 13.36%


# 6. Treinar modelo de Regressão Logística

In [12]:
modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_treino, y_treino)

print("Modelo treinado com sucesso!")

Modelo treinado com sucesso!


# 7. Fazer previsões e calcular métricas

In [13]:
predicoes = modelo.predict(X_teste)

acuracia = accuracy_score(y_teste, predicoes)
precisao = precision_score(y_teste, predicoes)
recall = recall_score(y_teste, predicoes)

print("=== Resultados da Classificação de Spam ===\n")
print(f"Acurácia: {acuracia:.4f} ({acuracia*100:.2f}%)")
print(f"Precisão: {precisao:.4f} ({precisao*100:.2f}%)")
print(f"Recall: {recall:.4f} ({recall*100:.2f}%)")

=== Resultados da Classificação de Spam ===

Acurácia: 0.9767 (97.67%)
Precisão: 0.9920 (99.20%)
Recall: 0.8322 (83.22%)


# 8. Matriz de confusão

In [14]:
matriz = confusion_matrix(y_teste, predicoes)

print("=== Matriz de Confusão ===\n")
print("                 Previsto Ham  Previsto Spam")
print(f"Real Ham (não spam):     {matriz[0,0]:>5}          {matriz[0,1]:>5}")
print(f"Real Spam:               {matriz[1,0]:>5}          {matriz[1,1]:>5}")

print("\n--- Interpretação ---")
print(f"Verdadeiros Negativos (VN): {matriz[0,0]} → modelo acertou que NÃO era spam")
print(f"Falsos Positivos (FP):      {matriz[0,1]} → modelo errou, disse SPAM mas NÃO era")
print(f"Falsos Negativos (FN):      {matriz[1,0]} → modelo errou, disse NÃO SPAM mas era spam")
print(f"Verdadeiros Positivos (VP): {matriz[1,1]} → modelo acertou que era spam")

=== Matriz de Confusão ===

                 Previsto Ham  Previsto Spam
Real Ham (não spam):       965              1
Real Spam:                  25            124

--- Interpretação ---
Verdadeiros Negativos (VN): 965 → modelo acertou que NÃO era spam
Falsos Positivos (FP):      1 → modelo errou, disse SPAM mas NÃO era
Falsos Negativos (FN):      25 → modelo errou, disse NÃO SPAM mas era spam
Verdadeiros Positivos (VP): 124 → modelo acertou que era spam


# 9. Testar diferentes proporções de treino/teste

In [15]:
print("=== Testando diferentes proporções de treino/teste ===\n")

proporcoes = [0.5, 0.6, 0.7, 0.8, 0.9]

for prop in proporcoes:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=1-prop, random_state=42, stratify=y)
    m = LogisticRegression(max_iter=1000)
    m.fit(X_tr, y_tr)
    pred = m.predict(X_te)

    acc = accuracy_score(y_te, pred)
    prec = precision_score(y_te, pred)
    rec = recall_score(y_te, pred)

    print(f"Treino {prop*100:.0f}% / Teste {(1-prop)*100:.0f}% → Acurácia: {acc:.4f}, Precisão: {prec:.4f}, Recall: {rec:.4f}")

=== Testando diferentes proporções de treino/teste ===

Treino 50% / Teste 50% → Acurácia: 0.9734, Precisão: 0.9934, Recall: 0.8070
Treino 60% / Teste 40% → Acurácia: 0.9749, Precisão: 0.9919, Recall: 0.8194
Treino 70% / Teste 30% → Acurácia: 0.9749, Precisão: 0.9892, Recall: 0.8214
Treino 80% / Teste 20% → Acurácia: 0.9767, Precisão: 0.9920, Recall: 0.8322
Treino 90% / Teste 10% → Acurácia: 0.9749, Precisão: 0.9841, Recall: 0.8267


# 10. Mostrar exemplos de erros

In [16]:
print("=== Exemplos de mensagens classificadas erroneamente ===\n")

# Pegar os índices dos erros
erros = np.where(predicoes != y_teste)[0]
df_teste = df.iloc[X_teste_indices] if 'X_teste_indices' in dir() else df.sample(len(X_teste), random_state=42)

# Recuperar os índices originais do teste
_, X_teste_indices = train_test_split(range(len(df)), test_size=0.2, random_state=42, stratify=y)
df_teste = df.iloc[X_teste_indices].reset_index(drop=True)

print("Falsos Positivos (mensagem boa classificada como spam):")
fps = np.where((predicoes == 1) & (y_teste == 0))[0]
for i in fps[:3]:
    print(f"- {df_teste.iloc[i]['message'][:100]}...")

print("\nFalsos Negativos (mensagem spam classificada como boa):")
fns = np.where((predicoes == 0) & (y_teste == 1))[0]
for i in fns[:3]:
    print(f"- {df_teste.iloc[i]['message'][:100]}...")

=== Exemplos de mensagens classificadas erroneamente ===

Falsos Positivos (mensagem boa classificada como spam):
- I'm always on yahoo messenger now. Just send the message to me and i.ll get it you may have to send ...

Falsos Negativos (mensagem spam classificada como boa):
- FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it s...
- Dear Voucher Holder 2 claim your 1st class airport lounge passes when using Your holiday voucher cal...
- ringtoneking 84484...


# 11. Respostas
### Pergunta 1: Qual foi a acurácia, precisão e recall obtidos?

**Resposta:** Com a proporção padrão de 80% treino e 20% teste, obtive:

- **Acurácia:** 0.9767 (97,67%)
- **Precisão:** 0.9920 (99,20%)
- **Recall:** 0.8322 (83,22%)

O modelo teve um desempenho excelente, acertando quase 98% das classificações no total.

---

### Pergunta 2: O que cada métrica significa neste contexto (filtro de spam)?

**Resposta:**

- **Acurácia (97,67%):** O modelo acertou a classificação em quase 98% das mensagens. Isso significa que, de cada 100 SMS, ele classificou corretamente cerca de 98.

- **Precisão (99,20%):** Quando o modelo disse que uma mensagem era SPAM, ele estava correto em 99,2% das vezes. Apenas 0,8% das mensagens que foram para a caixa de spam eram, na verdade, mensagens normais (ham). Isso é excelente para um filtro de spam.

- **Recall (83,22%):** O modelo identificou 83,22% de todos os spams que existiam. Isso significa que cerca de 16,78% dos spams escaparam e foram parar na caixa de entrada.

---

### Pergunta 3: Para um filtro de spam, qual métrica você acha mais importante? Por quê?

**Resposta:** Para um filtro de spam de SMS, a métrica mais importante é a **Precisão**.

**Justificativa:** Em um filtro de spam, o pior cenário é um **Falso Positivo (FP)** – uma mensagem normal (ham) sendo classificada como spam e indo para a lixeira. Isso pode fazer com que o usuário perca mensagens importantes como:
- Códigos de verificação bancária
- Confirmações de compra
- Recados de familiares ou trabalho

Já um **Falso Negativo (FN)** – um spam que cai na caixa de entrada – é menos grave, pois o usuário pode simplesmente ignorar ou deletar a mensagem. É um incômodo menor do que perder uma mensagem importante.

Portanto, priorizar a **precisão** garante que quase nunca uma mensagem boa seja filtrada por engano. No meu resultado, a precisão de 99,2% é excelente – apenas 0,8% das mensagens boas seriam perdidas.

---

### Pergunta 4: O que aconteceu quando você mudou a proporção de treino/teste?

**Resposta:** Ao variar a proporção de treino/teste, observei o seguinte:

| Proporção | Acurácia | Precisão | Recall | Observação |
|-----------|----------|----------|--------|-------------|
| 50% / 50% | 97,34% | 99,34% | 80,70% | Menor recall |
| 60% / 40% | 97,49% | 99,19% | 81,94% | Melhorou recall |
| 70% / 30% | 97,49% | 98,92% | 82,14% | Recall continua subindo |
| 80% / 20% | 97,67% | 99,20% | 83,22% | Melhor resultado geral |
| 90% / 10% | 97,49% | 98,41% | 82,67% | Precisão caiu um pouco |

**Principais observações:**

1. **Acurácia** se manteve estável entre 97,34% e 97,67% – variação muito pequena. O dataset é grande o suficiente (mais de 5500 mensagens) para que mesmo com 50% para treino o modelo já aprenda bem.

2. **Precisão** ficou sempre acima de 98%, chegando a 99,34% com 50% de treino. Isso é excelente – o modelo raramente classifica uma mensagem boa como spam.

3. **Recall** foi a métrica que mais variou: começou em 80,70% e subiu para 83,22% com 80% de treino. Isso mostra que **mais dados de treino ajudam o modelo a detectar mais spams**.

4. O melhor equilíbrio foi com **80% treino / 20% teste**: maior acurácia e recall, mantendo precisão alta.

**Conclusão:** A proporção 80/20 se mostrou a melhor para este dataset, equilibrando bom desempenho e quantidade suficiente de dados para teste confiável.

---

### Pergunta 5 (extra): Qual foi a melhor proporção e por quê?

**Resposta:** A melhor proporção foi **80% treino / 20% teste** porque:

- Acurácia mais alta (97,67%)
- Recall mais alto (83,22%) – detecta mais spams
- Precisão excelente (99,20%)
- É o padrão da indústria para datasets de tamanho médio

Com 90% de treino, o recall caiu um pouco e a precisão também, provavelmente porque a base de teste ficou muito pequena (apenas 10% dos dados), tornando a avaliação menos confiável.

# 12. Palavras que mais indicam spam

In [17]:
# Descobrir quais palavras são mais associadas ao spam
coeficientes = modelo.coef_[0]
palavras = vectorizer.get_feature_names_out()

palavras_spam = sorted(zip(coeficientes, palavras), key=lambda x: x[0], reverse=True)

print("=== Palavras que mais indicam SPAM ===\n")
for coef, palavra in palavras_spam[:15]:
    print(f"{palavra}: {coef:.4f}")

print("\n=== Palavras que mais indicam NÃO SPAM (HAM) ===\n")
for coef, palavra in palavras_spam[-15:][::-1]:
    print(f"{palavra}: {coef:.4f}")

=== Palavras que mais indicam SPAM ===

uk: 2.5255
service: 2.0461
txt: 1.9144
new: 1.8742
150p: 1.8664
message: 1.8323
claim: 1.7989
mobile: 1.7745
ringtone: 1.7420
www: 1.6827
urgent: 1.6155
reply: 1.6123
won: 1.6016
free: 1.5769
dating: 1.5679

=== Palavras que mais indicam NÃO SPAM (HAM) ===

gt: -1.1325
lt: -1.1174
fullonsms: -1.0664
ll: -0.8760
later: -0.7428
ok: -0.7410
da: -0.7242
got: -0.7186
people: -0.7091
hey: -0.6934
sorry: -0.6895
way: -0.6787
happy: -0.6728
home: -0.6634
lor: -0.6292
